# 时间旅行

时间旅行：允许你从某一个检查点开始重新运行直到图结束。

支持两种模型的时间旅行：

- Replay: 从某个检查点的状态开始重新运行
- Fork：和 Replay 一样，只是运行时可以修改检查点状态

<img src="./assets/时间旅行.svg?1">

## LangGraph API

In [ ]:
from langgraph_python.graphs.core_agent_graph import build_graph
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnableConfig
from langgraph_python.states.core_agent_state import ContextSchema
from langgraph.graph import START

context = ContextSchema(
    system_prompt="尽可能简短的回答问题，不要长篇大论",
    tools=[] # 不使用工具
)
checkpointer = InMemorySaver()
config:RunnableConfig = {
    "configurable":{
        "thread_id": "time_travel"
    }
}

graph = build_graph().compile(checkpointer=checkpointer)

In [ ]:
# 第一次运行
await graph.ainvoke(
    input={
        "messages": [
            HumanMessage("月亮离地球有多远")
    ]},
    config=config,
    context=context
)

### 查看历史记录

In [ ]:
history = list(graph.get_state_history(config))
# 打印每个检查点的 checkpoint_id 及它的父检查点
for i, state in enumerate(history):
    cid = state.config["configurable"]["checkpoint_id"] # type: ignore
    parent = (
        state.parent_config["configurable"]["checkpoint_id"] # type: ignore
        if state.parent_config
        else None
    )
    print(f"{i:>2}  checkpoint_id={cid}  parent={parent}")
[[msg.text for msg in h.values["messages"]] for h in history]

In [ ]:
before_start = history[2].config
after_start = history[1].config
before_start, after_start

### Replay

In [ ]:
await graph.ainvoke(
    input=None, # 不要有任何输入
    config=after_start,
    context=context
)

### fork

Fork 与 Replay 的区别：可以先**修改检查点状态**再运行。

In [ ]:
# 修改状态
fork_config = graph.update_state(
    config=before_start,
    values={"messages": [HumanMessage("太阳距离地球有多远")]},
    as_node=START,
)
fork_config

In [ ]:
# replay
await graph.ainvoke(
    input=None, # 不要有任何输入
    config=fork_config,
    context=context
)

### 最佳实践

- 刷新 AI 输出：找到AI结果之前的检查点，直接 replay 即可
- 修改内容：找到内容产出节点上一个检查点，指定`as_node`为内容产出节点，修改内容，然后从该检查点 replay

## Agent Server

Agent Server 上运行的图同样基于检查点，因此时间旅行（Replay / Fork）可以直接通过 SDK 操作。

与 LangGraph API 的区别只在于：历史、状态与检查点都存在服务器上，通过 `client.threads` 查询与修改，再通过 `client.runs` 从指定检查点恢复运行。

API文档地址：http://localhost:2024/docs

### 准备工作

获取一个 `agent` Assistant，创建线程并运行一次。`agent` 图会调用搜索工具，一次运行会产生多个检查点：

In [ ]:
from langgraph_sdk import get_client

# 连接本地 Agent Server
client = get_client(url="http://localhost:2024")
client

In [16]:
# 删除所有线程，免得看晕了
threads = await client.threads.search()
for t in threads:
    await client.threads.delete(thread_id=t['thread_id'])

In [17]:
# 获取一个已注册的 agent Assistant
assistant_id = "71463029-b316-4f54-96a8-e7a65f00af62"

In [18]:
# 创建线程，并运行一次产生多个检查点
thread = await client.threads.create(metadata={"__name__": "时间旅行"})
thread_id = thread["thread_id"]

await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={"messages": [{"role": "user", "content": "地球为什么是圆的？"}]},
)
thread_id

'019ff9af-dce9-7e13-ba04-f669f16754d0'

In [19]:
history = await client.threads.get_history(thread_id, limit=20)
before_start = {}
after_start = {}
for i, state in enumerate(history):
    step = state["metadata"].get("step") # type: ignore
    checkpoint_id = state["checkpoint"]["checkpoint_id"]
    print(f"{i:>2}  step={step:>3}  next={state['next']}  checkpoint_id={checkpoint_id}")
    if step == -1:
        before_start = state["checkpoint"]
    elif step == 0:
        after_start = state["checkpoint"]

before_start, after_start

 0  step=  1  next=[]  checkpoint_id=1f196db7-a9bf-6ce6-8001-256a3d02170d
 1  step=  0  next=['model']  checkpoint_id=1f196db7-9a47-65a2-8000-1f0fd43da987
 2  step= -1  next=['__start__']  checkpoint_id=1f196db7-9a44-6a78-bfff-658db76986bd


({'checkpoint_id': '1f196db7-9a44-6a78-bfff-658db76986bd',
  'thread_id': '019ff9af-dce9-7e13-ba04-f669f16754d0',
  'checkpoint_ns': ''},
 {'checkpoint_id': '1f196db7-9a47-65a2-8000-1f0fd43da987',
  'thread_id': '019ff9af-dce9-7e13-ba04-f669f16754d0',
  'checkpoint_ns': ''})

### Replay

> POST /threads/{thread_id}/runs/wait

In [ ]:
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input=None,
    checkpoint=after_start # type: ignore
)

result

### fork

In [ ]:
from langgraph.graph import START
updated = await client.threads.update_state(
    thread_id,
    values={
        "messages": [
            {"role":"user", "content":"月球到地球距离是多少？"}
        ]
    },
    checkpoint=before_start, # type: ignore
    as_node=START,
)
updated_checkpoint = updated["checkpoint"]
updated_checkpoint

In [ ]:
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input=None,
    checkpoint=updated_checkpoint
)
result

## 子图

子图无法做时间旅行

## [选修]魔法版本实现

解决 `LangSmith Studio` 显示的 bug (也可能是`LangGraph`的设计缺陷)

<img src="./assets/时间旅行_魔法.svg">

### LangGraph API

#### replay


In [ ]:
# copy 模式: 魔法
# 1. 先 copy 一个检查点出来

fork_config = graph.update_state(
    after_start, # type: ignore
    values=None, # 这里必须要传None
    as_node="__copy__" # 固定值
)
fork_config

In [ ]:
# copy 模式：魔法
# 2. 使用copy的检查点运行
await graph.ainvoke(
    input=None, # 不要有任何输入
    config=fork_config,
    context=context
)

#### fork


In [ ]:
fork_config = graph.update_state(
    before_start, # type: ignore
    values=[
        [
            {"messages": [HumanMessage("太阳距离地球有多远？")]},
            START, # 在这里指定新状态来自于哪个节点
        ]
    ],
    as_node="__copy__", # 在这里用魔法
)
fork_config

In [ ]:
# 从修改后的检查点继续运行（next 为空，运行不会再有新输出，直接返回当前状态）
await graph.ainvoke(
    input=None, # 不要有任何输入
    config=fork_config,
    context=context
)

### Agent Server

#### replay


In [ ]:
# 使用copy模式
# 1. copy 检查点

updated = await client.threads.update_state(
    thread_id=thread_id,
    values=None,
    checkpoint=after_start, # type: ignore
    as_node="__copy__",
)
updated_checkpoint = updated["checkpoint"]
updated_checkpoint

In [ ]:
# 使用copy模式
# 2. replay

result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input=None,
    checkpoint=updated_checkpoint
)

result

#### Fork


In [20]:
updated = await client.threads.update_state(
    thread_id,
    values=[
        [
            {"messages": [{"role":"user", "content":"月球到地球距离是多少？"}]},
            START,
        ] # type: ignore
    ],
    checkpoint=before_start, # type: ignore
    as_node="__copy__",
)
updated_checkpoint = updated["checkpoint"]
updated_checkpoint

{'thread_id': '019ff9af-dce9-7e13-ba04-f669f16754d0',
 'checkpoint_ns': '',
 'checkpoint_id': '1f196db8-33e9-600c-8001-80478af3def6'}

In [21]:
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input=None,
    checkpoint=updated_checkpoint
)
result

{'messages': [{'content': '月球到地球距离是多少？',
   'additional_kwargs': {},
   'response_metadata': {},
   'type': 'human',
   'name': None,
   'id': '908f65df-0938-40e8-b4bc-cd71917ed625'},
  {'content': '平均距离约为 38.44 万公里。',
   'additional_kwargs': {},
   'response_metadata': {'id': 'msg_cfe7a8f6-955f-9dd6-ab54-dea51c6a3986',
    'container': None,
    'model': 'qwen3.7-plus',
    'stop_details': None,
    'stop_reason': 'end_turn',
    'stop_sequence': None,
    'usage': {'cache_creation': None,
     'cache_creation_input_tokens': 0,
     'cache_read_input_tokens': 0,
     'inference_geo': None,
     'input_tokens': 30,
     'output_tokens': 12,
     'output_tokens_details': None,
     'server_tool_use': None,
     'service_tier': None,
     'prompt_tokens_details': {'cached_tokens': 0}},
    'model_name': 'qwen3.7-plus',
    'model_provider': 'anthropic'},
   'type': 'ai',
   'name': None,
   'id': 'lc_run--019ff9b0-2ef1-7793-90c4-8b7364c1f605-0',
   'tool_calls': [],
   'invalid_tool_call